In [13]:
import base64
import json
import os
import random
from openai import OpenAI
from pydantic import BaseModel
from typing import List, Dict

定义图像编码函数
- 4o系列：请使用Base64编码将图片转换为字符串格式。
- QwenVL系列：可以直接发送原始图片数据，无需进行编码。

In [14]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

定义QA对

In [15]:
class QA_Pair(BaseModel):
    question: str
    answer: str



##### 类定义：`Cmanager`
`Cmanager`类是一个用于管理网络拓扑图像和相关操作的类。它提供了初始化客户端、设置拓扑图像路径、提取实体、构建问答对以及保存数据到JSON文件的功能。
属性：
- `client`: 用于与API通信的客户端对象。
- `topology_image_path`: 存储拓扑图像文件路径的字符串。
方法：
1. `__init__(self, api_base: str, api_key: str)`：
   - 初始化方法，接收两个参数：`api_base`（API的基础URL）和`api_key`（API的密钥）。
   - 创建客户端实例并存储在`self.client`中。
   - 初始化`topology_image_path`为空字符串。
2. `get_client(self, base_url, api_key)`：
   - 私有方法，用于创建并返回一个`OpenAI`客户端实例。
   - 参数：`base_url`和`api_key`。
3. `set_topology_image(self, image_path: str)`：
   - 设置拓扑图像文件路径。
   - 参数：`image_path`（图像文件的路径）。
4. `extract_entities(self) -> List[str]`：
   - 提取拓扑图像中的网络实体。
   - 检查`topology_image_path`是否已设置，如果没有，则抛出`ValueError`。
   - 将图像编码为base64格式，并构建系统内容字符串。
   - 使用`gpt-4o-mini`模型创建聊天完成请求，提取实体并返回实体列表。
5. `build_qa_pairs(self, entities: List[str], num_pairs: int = 10) -> List[Dict[str, str]]`：
   - 基于提取的实体构建问答对。
   - 参数：`entities`（实体列表），`num_pairs`（要生成的问答对数量，默认为10）。
   - 随机选择实体和问题模板，构建问题并使用`gpt-4o-mini`模型生成答案。
   - 返回包含问题和答案的字典列表。
6. `save_to_json(self, entities: List[str], qa_pairs: List[Dict[str, str]], filename: str)`：
   - 将实体和问答对保存为JSON文件。
   - 参数：`entities`（实体列表），`qa_pairs`（问答对列表），`filename`（要保存的文件名）。

##### 注意事项：
1. **图像路径设置**：在调用`extract_entities`和`build_qa_pairs`方法之前，必须通过`set_topology_image`方法设置拓扑图像路径。
2. **图像编码**：`extract_entities`和`build_qa_pairs`方法中使用了`encode_image`函数对图像进行base64编码，确保该函数已正确实现。
3. **问答对生成**：`build_qa_pairs`方法默认生成10个问答对，可以根据需要调整`num_pairs`参数。
4. **JSON文件保存**：`save_to_json`方法将实体和问答对保存为JSON文件，确保指定的`filename`路径可写。
5. **随机性**：`build_qa_pairs`方法中使用了随机选择实体和问题模板，这可能导致每次生成的问答对不同。

In [16]:
## 定义Cmanager类
class Cmanager:
    def __init__(self, api_base: str, api_key: str):
        self.client = self.get_client(api_base, api_key)
        self.topology_image_path = ""

    def get_client(self, base_url, api_key):
        client = OpenAI(base_url=base_url, api_key=api_key)
        return client

    def set_topology_image(self, image_path: str):
        self.topology_image_path = image_path

    # STEP1 抽取实体
    def extract_entities(self) -> List[str]:
        if not self.topology_image_path:
            raise ValueError("Topology image path is not set.")
        base64_image = encode_image(self.topology_image_path)

        content_system = (
            "You are a network topology entity extraction expert specialized in identifying named key network elements."
            "You will receive an image depicting a network topology."
            "Your task is to identify **named** core network elements from the image."
            "Extract a maximum of five (**five (5) most crucial named core network elements**) of the core network elements from the named ones."
            "If there are less than five named core network elements, output all of them."
            "If the image does not contain at least one named core network element, output an empty list: `[]`."
            "1. Core Network Elements:"
            "  - Core network elements are essential components like routers, switches, servers, firewalls, etc., that have a distinct name in the image."
            "2. Output Format:"
            "  - Output the core network element **names** (not the types) as a list of strings. Do not use any other format."
            "  - Do not include any explanations, analysis, or additional information."
            f"  - Example output for an image with named elements: ['R1', 'SW1', 'ServerA']"
            f"  - Example output for an image with no named elements: []"
            "3. Important Notes:"
            "  - **Focus solely on named entities. Unnamed elements should be ignored.**"
            "  - Extract a **maximum of five** core network element names(different names)."
            "  - Output the extracted names **exactly as they appear** in the image."
            "  - Ensure the output is a valid list."
        )

        completion = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": content_system},
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "Topology Image"},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            },
                        },
                    ],
                },
            ],
            temperature=0.2,
        )

        response_content = completion.choices[0].message.content.strip()

        list_str = response_content[2:-2]
        entities = [entity.strip("'") for entity in list_str.split(", ")]
        print("Raw response:", response_content)
        return entities

    # STEP2 生成QA对

    def build_qa_pairs(
        self, entities: List[str], num_pairs: int = 10
    ) -> List[Dict[str, str]]:
        qa_pairs = []
        base64_image = encode_image(self.topology_image_path)

        # question_templates = [
        #     # TODO 增加QA类型，不仅仅局限在两个节点之间的联系
        #     "How are {entity1} and {entity2} connected? Describe any dependencies between them.",
        #     "Explain the relationship between these network elements: {entities}.",
        #     "Based on the topology, which network elements can be considered a group? What common characteristics do they share?",
        #     "If you need to isolate {entity1} and {entity2} into separate network zones, how would you achieve this?",
        #     "If {entity1} were to fail, which other network elements would be affected and how?",
        #     "What is the path data takes when traveling from {entity1} to {entity2}?",
        #     "If a new network element needs to be added to connect {entity1} and {entity2}, where would you place it and why?",
        #     "Identify any potential bottlenecks or single points of failure in the connection between {entity1} and {entity2}.",
        #     "Describe the security implications of the connection between {entity1} and {entity2}.",
        #     "How does the placement of {entity1} and {entity2} affect the overall performance and reliability of the network?",
        #     "Assuming {entity1} is a web server and {entity2} is a client, describe the communication flow between them.",
        #     "If you wanted to increase the bandwidth between {entity1} and {entity2}, what changes could you make to the topology?",
        #     "What protocols are likely being used for communication between {entity1} and {entity2}?",
        #     "How can you monitor the traffic flow and performance of the connection between {entity1} and {entity2}?",
        #     "If there is a security breach at {entity1}, how could it potentially impact {entity2}?",
        # ]
        question_templates = [
            "What is the overall topology of this network? Is it a star, bus, ring, mesh, or hybrid topology?",
            "What are the key network segments in this topology, and what are their purposes?",
            "Describe the role of the {entity} in this network and how it interacts with other entities.",
            "How would you implement a backup strategy for the {entity}, considering its relations with other critical entities in the network?",
            "What are some potential security vulnerabilities in this network, particularly concerning the interactions between {entity1} and {entity2}, and how would you address them?",
        ]

        if len(entities) > 0:
            question_templates = [
                "What is the overall topology of this network? Is it a star, bus, ring, mesh, or hybrid topology?",
                "What are the key network segments in this topology, and what are their purposes?",
                "Describe the role of the {entity} in this network and how it interacts with other entities.",
                "How would you implement a backup strategy for the {entity}, considering its relations with other critical entities in the network?",
                "What are some potential security vulnerabilities in this network, particularly concerning the interactions between {entity1} and {entity2}, and how would you address them?",
            ]
        else:
            question_templates = [
                "What is the overall topology of this network? Is it a star, bus, ring, mesh, or hybrid topology?",
                "What are the key network segments in this topology, and what are their purposes?",
                "Describe the role of the main network elements in this network and how they interact with each other.",
                "How would you implement a backup strategy for the main network elements, considering their relations with each other?",
                "What are some potential security vulnerabilities in this network, particularly concerning the interactions between the main network elements, and how would you address them?",
                ]
                

        # question_templates = [
        #     # TODO 增加QA类型，不仅仅局限在两个节点之间的联系
        #     "What is the overall topology of this network? Is it a star, bus, ring, mesh, or hybrid topology?",
        #     "What are the key network segments in this topology, and what are their purposes?",
        #     "Describe the role of the {entity} in this network.",
        #     "How would you implement a backup strategy for the {entity}?",
        #     "What are some potential security vulnerabilities in this network, and how would you address them?",
        #     "How would you ensure high availability for critical services on this network?",
        #     "What are the advantages and disadvantages of using a {entity} in this topology?",
        #     "How would you perform network troubleshooting in case of a connectivity issue between {entity1} and {entity2}?",
        #     "How would you configure VLANs in this network to improve security and performance?",
        #     "What kind of routing protocols (e.g., RIP, OSPF, BGP) might be used in this network?",
        #     "How would you implement access control policies for different types of users in this network?",
        #     "What are the main differences between the {entity1} and {entity2} in this topology?",
        #     "How would you measure network latency between different segments of this network?",
        #     "What is the maximum bandwidth capacity of the link between {entity1} and {entity2}?",
        #     "How would you monitor the health of the {entity} in this topology?",
        #     "What are the potential consequences of a failure in the {entity}?",
        #     "Describe the data flow path for a user accessing the internet from a client within this network.",
        #     "How would you implement a guest network in this topology?",
        #     "What are the best practices for managing and maintaining this network?",
        #     "How would you analyze network traffic patterns in this topology?",
        #     "Explain the purpose of the firewall in this network.",
        #     "What are the potential challenges in managing this network, and how would you overcome them?",
        #     "How would you improve the overall performance of this network?",
        #     "What are some common troubleshooting steps for a network outage in this topology?",
        #     "What considerations should be made when adding a new device to this network?",
        #     "How would you implement load balancing across multiple servers in this network?",
        #     "What are the best practices for network security in this topology?",
        #     "How would you ensure data integrity in this network?",
        #     "How would you configure network address translation (NAT) in this topology?",
        #     "How would you implement a VPN in this network?",
        #     "How do the network devices in this topology interact with each other?",
        #     "What are the main protocols used in this network?",
        #     "What are the potential use cases for this network setup?",
        #     "How would you segment this network into multiple subnets?",
        #     "What are the potential benefits of using cloud-based services in conjunction with this network?",
        #     "How could you leverage network automation tools to manage this infrastructure?",
        #     "Identify the most critical devices in this topology and explain why they are critical.",
        #     "Describe how the network traffic is managed and prioritized in this network.",
        #     "Summarize the topology diagram.",
        # ]

        num_pairs = min(num_pairs, len(question_templates))

        for _ in range(num_pairs):
            # 随机选取几个问题模版
            question_template = random.choice(question_templates)

            # 随机选取几个实体
            num_entities_in_question = random.randint(1, len(entities))

            sampled_entities = random.sample(entities, num_entities_in_question)

            # 格式化使用的实体--避免输出的entity错误
            if "{entity}" in question_template:
                question = question_template.format(entity=sampled_entities[0])
            elif "{entity1}" in question_template and "{entity2}" in question_template:
                if len(sampled_entities) >= 2:
                    question = question_template.format(
                        entity1=sampled_entities[0], entity2=sampled_entities[1]
                    )
                else:
                    question = question_template.format(
                        entity1=sampled_entities[0], entity2=sampled_entities[0]
                    )
            else:
                question = question_template

            content_system = (
                "You are a network topology expert assistant."
                "Your task is to answer questions about a network topology based on a provided image and a list of key node entities."
                "Instructions:"
                "1. Carefully examine the topology image and corresponding key node entities: Understand the connections and relationships between the key node entities, and use the entity names to identify components."
                "2. Answer the question accurately and concisely: Provide a clear and direct answer to the question without making assumptions or introducing external information."
                "3. Output your answer as a string."
                "Topology Image:"
                "Entities:"
                f"{entities}"
                "Question:"
                f"{question}"
            )

            completion = self.client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": content_system},
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": "Topology Image"},
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/jpeg;base64,{base64_image}"
                                },
                            },
                            {"type": "text", "text": f"{question}"},
                        ],
                    },
                ],
                temperature=0.2,
                max_tokens=4095,
            )
            answer = completion.choices[0].message.content.strip()

            qa_pairs.append({"question": question, "answer": answer})

        return qa_pairs

    def save_to_json(
        self, entities: List[str], qa_pairs: List[Dict[str, str]], filename: str
    ):
        data = {"entities": entities, "qa_pairs": qa_pairs}
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4, ensure_ascii=False)

初始化Cmanager实例并设置网络拓扑图像路径

In [17]:
api_base = "https://neudm.zeabur.app/v1"
api_key = "sk-T05m0OqxOgKUjErs8c231e1c02E24573A17977F5E839E91c"
cmanager = Cmanager(api_base=api_base, api_key=api_key)

In [18]:
sub_paths = ["normal"]
data_path = "data/images"
res_path = "data/result"
for sub in sub_paths:
    topology_image_path = os.path.join(data_path, sub)

    # 列出目录下的所有文件
    for filename in os.listdir(topology_image_path):
        # 构建完整的文件路径
        file_path = os.path.join(topology_image_path, filename)
        if os.path.isfile(file_path):
            cmanager.set_topology_image(file_path)
            try:
                entities = cmanager.extract_entities()
                if entities:
                    print("Extracted network element entities:", entities)
                    qa_pairs = cmanager.build_qa_pairs(entities)
                    print("Built QA pairs:", qa_pairs)

                    # 构建结果保存的目录路径
                    result_dir = os.path.join(res_path, sub)
                    if not os.path.exists(result_dir):
                        os.makedirs(result_dir)

                    # 构建结果文件的完整路径
                    image_name_without_ext = os.path.splitext(filename)[0]
                    output_filename = f"{image_name_without_ext}.json"
                    output_file_path = os.path.join(result_dir, output_filename)

                    cmanager.save_to_json(entities, qa_pairs, output_file_path)
                    print(f"Results saved to {output_file_path}")
            except Exception as e:
                print(f"Error processing file {file_path}: {e}")

Raw response: ['路由器', '防火墙', '主交换机', '交换机', '应用服务器']
Extracted network element entities: ['路由器', '防火墙', '主交换机', '交换机', '应用服务器']
Built QA pairs: [{'question': 'What is the overall topology of this network? Is it a star, bus, ring, mesh, or hybrid topology?', 'answer': 'The overall topology of this network is a hybrid topology. It combines elements of star and bus topologies, as there are multiple switches connected to a central switch, and PCs connected to those switches.'}, {'question': 'How would you implement a backup strategy for the 主交换机, considering its relations with other critical entities in the network?', 'answer': "To implement a backup strategy for the 主交换机 (main switch), consider the following steps:\n\n1. **Redundant Connections**: Establish redundant connections between the 主交换机 and other switches (交换机) to ensure that if one link fails, traffic can still flow through an alternative path.\n\n2. **Spanning Tree Protocol (STP)**: Enable STP on all switches to prevent loops a

KeyboardInterrupt: 